# 🚋 Muni Schedule (Static GTFS) - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests **SFMTA Muni's** published schedule (GTFS) into Elasticsearch: every bus, Muni Metro, streetcar, and cable car stop (with geo location), every route, and - optionally - the timetable for one line.

**What this notebook does:**
1. Downloads the [Muni GTFS feed](https://www.sfmta.com/reports/gtfs-transit-data) - no API key required
2. Indexes stops into `muni_stops` (with `geo_point`) and routes into `muni_routes`
3. **(Optional)** indexes one route's scheduled stop times into `muni_stop_times`

---
> **Data source:** SFMTA GTFS (`muni_gtfs-current.zip`). Use is subject to the SFMTA Transit Data License Agreement included in the zip. For **live** Muni predictions, see the 511.org section of [bart_realtime.ipynb](bart_realtime.ipynb).

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests, io, zipfile, csv
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Download and open the GTFS zip

In [ ]:
GTFS_URL = 'https://muni-gtfs.apps.sfmta.com/data/muni_gtfs-current.zip'
resp = requests.get(GTFS_URL, timeout=120)
resp.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(resp.content))
print('✅ Downloaded GTFS. Files inside:')
print('  ', zf.namelist())

def read_csv(name):
    with zf.open(name) as f:
        return list(csv.DictReader(io.TextIOWrapper(f, encoding='utf-8-sig')))

## 3. Index the stops (with geo_point)

In [ ]:
STOPS_INDEX = 'muni_stops'
if es.indices.exists(index=STOPS_INDEX):
    es.indices.delete(index=STOPS_INDEX)
es.indices.create(index=STOPS_INDEX, body={'mappings': {'properties': {
    'stop_id':   {'type': 'keyword'},
    'stop_code': {'type': 'keyword'},   # the number printed on the stop sign / used for texting
    'stop_name': {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'wheelchair_boarding': {'type': 'keyword'},
    'location':  {'type': 'geo_point'},
}}})

stops = read_csv('stops.txt')
docs = []
for s in stops:
    try:
        loc = {'lat': float(s['stop_lat']), 'lon': float(s['stop_lon'])}
    except (KeyError, ValueError):
        continue
    docs.append({'stop_id': s['stop_id'], 'stop_code': s.get('stop_code'), 'stop_name': s['stop_name'],
                 'wheelchair_boarding': s.get('wheelchair_boarding'), 'location': loc})

helpers.bulk(es, [{'_index': STOPS_INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=STOPS_INDEX)
print(f'✅ Indexed {len(docs)} stops into {STOPS_INDEX}')

## 4. Index the routes
GTFS `route_type`: **0** = Muni Metro / streetcar, **3** = bus, **5** = cable car.

In [ ]:
ROUTES_INDEX = 'muni_routes'
ROUTE_TYPES = {'0': 'Metro / streetcar', '3': 'Bus', '5': 'Cable car'}
if es.indices.exists(index=ROUTES_INDEX):
    es.indices.delete(index=ROUTES_INDEX)
es.indices.create(index=ROUTES_INDEX, body={'mappings': {'properties': {
    'route_id':         {'type': 'keyword'},
    'route_short_name': {'type': 'keyword'},
    'route_long_name':  {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'route_type':       {'type': 'keyword'},
    'mode':             {'type': 'keyword'},
    'route_url':        {'type': 'keyword'},
}}})

routes = read_csv('routes.txt')
docs = [{'route_id': r['route_id'], 'route_short_name': r.get('route_short_name'),
         'route_long_name': r.get('route_long_name'), 'route_type': r.get('route_type'),
         'mode': ROUTE_TYPES.get(r.get('route_type'), 'Other'), 'route_url': r.get('route_url')}
        for r in routes]
helpers.bulk(es, [{'_index': ROUTES_INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=ROUTES_INDEX)
print(f'✅ Indexed {len(docs)} routes into {ROUTES_INDEX}')

## 5. Verify

In [ ]:
resp = es.search(index=ROUTES_INDEX, body={'size': 0, 'aggs': {'by_mode': {'terms': {'field': 'mode'}}}})
print('Routes by mode:')
for b in resp['aggregations']['by_mode']['buckets']:
    print(f"  {b['key']:20s} {b['doc_count']}")

# Stops within 250 m of the venue (33 New Montgomery St) - proves the geo_point works
resp = es.search(index=STOPS_INDEX, body={'size': 8,
    'query': {'geo_distance': {'distance': '250m', 'location': {'lat': 37.7873, 'lon': -122.4005}}},
    'sort': [{'_geo_distance': {'location': {'lat': 37.7873, 'lon': -122.4005}, 'order': 'asc', 'unit': 'm'}}],
})
print('\nMuni stops near the venue:')
for h in resp['hits']['hits']:
    print(f"  {h['sort'][0]:5.0f} m  {h['_source']['stop_name']}")

## 6. (Optional) Ingest one route's timetable
`stop_times.txt` is big (~100 MB, millions of rows), so we stream it and keep only the trips for one route. Pick any `route_short_name`: `N`, `J`, `T`, `14`, `38R`, `PM` (Powell-Mason cable car), ...

In [ ]:
ROUTE = 'N'   # route_short_name

route_ids = {r['route_id'] for r in routes if r.get('route_short_name') == ROUTE}
trips = {t['trip_id']: t for t in read_csv('trips.txt') if t['route_id'] in route_ids}
stop_names = {s['stop_id']: s['stop_name'] for s in stops}
print(f'{len(trips)} scheduled {ROUTE} trips')

sel = []
with zf.open('stop_times.txt') as f:
    for st in csv.DictReader(io.TextIOWrapper(f, encoding='utf-8-sig')):
        t = trips.get(st['trip_id'])
        if t:
            sel.append({'route': ROUTE, 'trip_id': st['trip_id'], 'service_id': t.get('service_id'),
                        'direction_id': t.get('direction_id'), 'headsign': t.get('trip_headsign'),
                        'stop_id': st['stop_id'], 'stop_name': stop_names.get(st['stop_id']),
                        'stop_sequence': int(st['stop_sequence']), 'arrival_time': st['arrival_time']})
print(f'{len(sel)} stop times for the {ROUTE}')

TIMES_INDEX = 'muni_stop_times'
if es.indices.exists(index=TIMES_INDEX):
    es.indices.delete(index=TIMES_INDEX)
es.indices.create(index=TIMES_INDEX, body={'mappings': {'properties': {
    'route': {'type': 'keyword'}, 'trip_id': {'type': 'keyword'}, 'service_id': {'type': 'keyword'},
    'direction_id': {'type': 'keyword'}, 'headsign': {'type': 'keyword'},
    'stop_id': {'type': 'keyword'}, 'stop_name': {'type': 'keyword'},
    'stop_sequence': {'type': 'integer'},
    'arrival_time': {'type': 'keyword'},   # HH:MM:SS, can run past 24:00 for late-night trips
}}})
helpers.bulk(es, [{'_index': TIMES_INDEX, '_source': d} for d in sel], raise_on_error=False)
es.indices.refresh(index=TIMES_INDEX)
print(f'✅ Indexed into {TIMES_INDEX}')

## 7. Going further with Mistral

- **Car-free trip planner:** give a Mistral agent function tools over `muni_stops` (geo) and `muni_routes`, plus [sf_parks_and_public_art](sf_parks_and_public_art.ipynb), so it can answer "How do I get to the Japanese Tea Garden without a car, and what else is near it?"
- **Accessibility helper:** filter stops by `wheelchair_boarding` and have Mistral explain the step-free option.
- **Structured outputs:** turn a free-text request ("I'm at Church and 24th, need to get to the Ferry Building") into origin/destination fields with a Mistral model.
- **Live data:** pair with [bart_realtime](bart_realtime.ipynb), which also shows how to pull live Muni predictions from 511.org.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`muni_stops` and `muni_routes` are ready (plus `muni_stop_times` if you ran step 6).

**To refresh the data**, re-run the notebook. Each index is dropped and recreated, so there are no duplicates.